In [1]:
#モジュールのインストール

!pip install requests
!pip install beautifulsoup4
!pip install urllib3
!pip install pandas

In [2]:
import requests
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from urllib.request import urlopen
from urllib.parse import urlparse

In [4]:
# ベースURL
base_url = 'https://jyuke-labo.com/koukoujyukentaisaku/hensachi/'

base_school_urls = []

try:
    req = requests.get(base_url)
    req.raise_for_status()
    req.encoding = req.apparent_encoding
except requests.exceptions.RequestException as e:
    print(f"エラーが発生しました: {str(e)}")
    exit()

# HTMLを解析
soup = BeautifulSoup(req.text, 'html.parser')
links = soup.find_all(class_="boxMapTable")

for link in links:
    a_tags = link.find_all('a', href=True)  # aタグを取得
    for a in a_tags:
        href = urljoin(base_url, a.get('href'))  # 相対URLを絶対URLに変換
        base_school_urls.append(href)
        time.sleep(1)

# 取得したURLリストを表示
print("\n=== 抽出したURL ===")
for url in base_school_urls:
    print(url)



=== 抽出したURL ===
https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/
https://jyuke-labo.com/koukoujyukentaisaku/aomori/
https://jyuke-labo.com/koukoujyukentaisaku/akita/
https://jyuke-labo.com/koukoujyukentaisaku/iwate/
https://jyuke-labo.com/koukoujyukentaisaku/yamagata/
https://jyuke-labo.com/koukoujyukentaisaku/fukushima/
https://jyuke-labo.com/koukoujyukentaisaku/miyagi/
https://jyuke-labo.com/koukoujyukentaisaku/fukuoka/
https://jyuke-labo.com/koukoujyukentaisaku/saga/
https://jyuke-labo.com/koukoujyukentaisaku/nagasaki/
https://jyuke-labo.com/koukoujyukentaisaku/kagoshima/
https://jyuke-labo.com/koukoujyukentaisaku/ooita/
https://jyuke-labo.com/koukoujyukentaisaku/kumamoto/
https://jyuke-labo.com/koukoujyukentaisaku/miyazaki/
https://jyuke-labo.com/koukoujyukentaisaku/okinawa/
https://jyuke-labo.com/koukoujyukentaisaku/tottori/
https://jyuke-labo.com/koukoujyukentaisaku/shimane/
https://jyuke-labo.com/koukoujyukentaisaku/okayama/
https://jyuke-labo.com/koukoujyukentaisaku/hiros

In [4]:
# 高校一覧ページのURLリスト（例: 各都道府県の高校一覧ページ）
base_school_urls = [
    "https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/",
    # 他の都道府県のURLを追加
]

# 全高校のURLを保存するリスト
All_Highschool_urls = []

# 各都道府県の高校一覧ページを巡回して高校のURLを取得
for school_url in base_school_urls:
    print(f"\n▶️ {school_url} の高校一覧を取得中...\n")
    
    try:
        res = requests.get(school_url)
        soup = BeautifulSoup(res.text, 'html.parser')

        # 国公立高校のURL取得
        Kouritu_urls = soup.find_all(class_="listUniv__list js-moreList_06")
        # 私立高校のURL取得
        Shiritu_urls = soup.find_all(class_="listUniv__list js-moreList_07")

        for section in (Kouritu_urls + Shiritu_urls):
            a_tags = section.find_all('a', href=True)
            for a in a_tags:
                href = urljoin(school_url, a.get('href'))
                All_Highschool_urls.append(href)
                print(f"✅ 高校URL取得: {href}")
                time.sleep(1)  # サーバー負荷軽減のため待機

    except requests.exceptions.RequestException as e:
        print(f"⚠️ {school_url} にアクセスできません: {e}")

# 高校ごとのデータを保存するリスト
highschool_data = []

# 各高校のページをスクレイピング
for highschool_url in All_Highschool_urls:
    print(f"\n🏫 {highschool_url} のデータ取得中...\n")

    try:
        req = requests.get(highschool_url)
        req.raise_for_status()
        req.encoding = req.apparent_encoding
        soup = BeautifulSoup(req.text, 'html.parser')

        ### ① 高校名の取得
        breadcrumb = soup.select("div.breadcrumb ol.breadcrumb__list li a")
        university_name = breadcrumb[-1].text.strip() if breadcrumb else "不明"

        ### ② 学科名の取得
        feature_section = soup.find("h2", id="feature")
        department_names = []
        feature_text = []

        if feature_section:
            for sibling in feature_section.find_next_siblings():
                if sibling.name == "p" and "＜教育課程＞" in sibling.get_text():
                    continue
                elif sibling.name == "p" and "＜学校行事＞" in sibling.get_text():
                    break
                else:
                    feature_text.append(sibling.get_text())

            # 【学科名】を抽出
            department_names = re.findall(r'【(.*?)】', '\n'.join(feature_text))

        ### ③ 学科説明の取得
        department_descriptions = {}
        for dep in department_names:
            pattern = rf'【{dep}】(.*?)\n'
            match = re.search(pattern, '\n'.join(feature_text))
            department_descriptions[dep] = match.group(1).strip() if match else "説明なし"

        ### ④ 偏差値の取得（「合格するには～」が出たら終了）
        hensachi_section = soup.find("h2", id="hensachi")
        hensachi_values = []
        if hensachi_section:
            for sibling in hensachi_section.find_next_siblings():
                if sibling.name == "p":
                    text = sibling.get_text().strip()
                    if "合格するには" in text:
                        break  # 偏差値の説明文が出たら終了
                    hensachi_values.append(text)

        ### ⑤ 進学実績の取得（複数の `grid_2_list` を考慮）
        shingaku_results = []
        shingaku_sections = soup.find_all("div", class_="grid_2_list")
        for section in shingaku_sections:
            for li in section.find_all("li"):
                shingaku_results.append(li.text.strip())

        # **取得したデータをリストに追加**
        highschool_data.append({
            "高校名": university_name,
            "URL": highschool_url,
            "学科名": ", ".join(department_names) if department_names else "学科情報なし",
            "学科説明": department_descriptions,
            "偏差値": ", ".join(hensachi_values) if hensachi_values else "偏差値情報なし",
            "進学実績": ", ".join(shingaku_results) if shingaku_results else "進学実績なし"
        })

        time.sleep(2)  # サーバー負荷軽減

    except requests.exceptions.RequestException as e:
        print(f"⚠️ {highschool_url} にアクセスできません: {e}")

# **Pandas DataFrame に変換**
df = pd.DataFrame(highschool_data)

# **データフレームを表示**
print("\n=== 全高校のデータ ===")
print(df)

# **CSVファイルとして保存（オプション）**
df.to_csv("highschool_data.csv", index=False, encoding="utf-8-sig")
print("\n✅ データを 'highschool_data.csv' に保存しました！")


▶️ https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/ の高校一覧を取得中...

✅ 高校URL取得: https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/sapporominamikoukou/
✅ 高校URL取得: https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/sapporokitakoukou/
✅ 高校URL取得: https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/sapporonisikoukou/
✅ 高校URL取得: https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/sapporohigasikoukou/
✅ 高校URL取得: https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/kusirokoryoukoukou/
✅ 高校URL取得: https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/sapporoasahiokakoukou/
✅ 高校URL取得: https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/sapporokokusaizyouhoukoukou/
✅ 高校URL取得: https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/muroransakaekoukou/
✅ 高校URL取得: https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/obihirohakuyoukoukou/
✅ 高校URL取得: https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/hakodatetyuubukoukou/
✅ 高校URL取得: https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/asahikaw

KeyboardInterrupt: 

In [20]:
# 高校のページをスクレイピング例！！
All_Highschool_urls = [
    "https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/sapporominamikoukou/",
    "https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/sapporokitakoukou/",
    "https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/hokuseigakuenzyosikoukou/"
]

# 高校ごとのデータを保存するリスト
highschool_data = []

for highschool_url in All_Highschool_urls:
    print(f"\n🏫 {highschool_url} のデータ取得中...\n")

    try:
        req = requests.get(highschool_url)
        req.raise_for_status()
        req.encoding = req.apparent_encoding
        soup = BeautifulSoup(req.text, 'html.parser')

        ### ① 高校名の取得
        breadcrumb = soup.select("div.breadcrumb ol.breadcrumb__list li a")
        university_name = breadcrumb[-1].text.strip() if breadcrumb else "不明"
        time.sleep(1)

        ### ② 学科名の取得（id="feature" が複数ある場合の対応）
        feature_sections = soup.find_all("h2", id="feature")

        # 適切な h2（＜教育課程＞ のテキストを含むもの）を選択
        feature_section = None
        for section in feature_sections:
            if "＜教育課程＞" in section.get_text():
                feature_section = section
                break  # 最初に見つかった適切なセクションを使用

        # 学科名を格納するリスト
        department_names = []
        department_descriptions = {}
        feature_text = []

        # feature_section が見つかった場合のみ処理
        if feature_section:
            for sibling in feature_section.find_next_siblings():
                if sibling.name == "p" and "＜教育課程＞" in sibling.get_text():
                    continue
                elif sibling.name == "p" and "＜学校行事＞" in sibling.get_text():
                    break
                else:
                    feature_text.append(sibling)

            # 【学科名】を抽出
            for elem in feature_text:
                match = re.match(r'【(.*?)】', elem.get_text())
                if match:
                    current_department = match.group(1).strip()
                    department_names.append(current_department)
                    department_descriptions[current_department] = ""

                # 直前に学科名があった場合、その説明として追加
                elif department_names:
                    department_descriptions[department_names[-1]] += " " + elem.get_text().strip()

        # **【】がない場合は <h2 id="hensachi"> の後の <p> から学科名を取得**
        if not department_names:
            hensachi_section = soup.find("h2", id="hensachi")
            if hensachi_section:
                for sibling in hensachi_section.find_next_siblings():
                    if sibling.name == "p":
                        text = sibling.get_text().strip()
                        if "合格するには" in text:
                            break  # 偏差値の説明文が出たら終了
                        
                        # 偏差値の行ごとに学科名を抽出（・と：の間の文字）
                        matches = re.findall(r'・(.*?)：', text)
                        department_names.extend(matches)

        # **【】がない場合、教育課程 ～ 学校行事 の間の全文章を学科説明とする**
        if not department_descriptions and feature_section:
            department_description_texts = []
            found_edu = False
            found_events = False

            for sibling in feature_section.find_next_siblings():
                if sibling.name == "p":
                    text = sibling.get_text().strip()
                    if "＜教育課程＞" in text:
                        found_edu = True
                        continue
                    elif "＜学校行事＞" in text:
                        found_events = True
                        break
                    elif found_edu and not found_events:
                        department_description_texts.append(text)

            department_descriptions["学科全体"] = "\n".join(department_description_texts) if department_description_texts else "説明なし"

        time.sleep(1)

        ### ④ 偏差値の取得（「合格するには～」が出たら終了）
        hensachi_values = []
        if hensachi_section:
            for sibling in hensachi_section.find_next_siblings():
                if sibling.name == "p":
                    text = sibling.get_text().strip()
                    if "合格するには" in text:
                        break  # 偏差値の説明文が出たら終了
                    hensachi_values.append(text)
        time.sleep(1)

        ### ⑤ 進学実績の取得（複数の `grid_2_list` を考慮）
        shingaku_results = []
        shingaku_sections = soup.find_all("div", class_="grid_2_list")
        for section in shingaku_sections:
            for li in section.find_all("li"):
                shingaku_results.append(li.text.strip())
        time.sleep(1)

        # **取得したデータをリストに追加**
        highschool_data.append({
            "高校名": university_name,
            "URL": highschool_url,
            "学科名": ", ".join(department_names) if department_names else "学科情報なし",
            "学科説明": department_descriptions if department_descriptions else "説明なし",
            "偏差値": ", ".join(hensachi_values) if hensachi_values else "偏差値情報なし",
            "進学実績": ", ".join(shingaku_results) if shingaku_results else "進学実績なし"
        })

        time.sleep(2)  # サーバー負荷軽減

    except requests.exceptions.RequestException as e:
        print(f"⚠️ {highschool_url} にアクセスできません: {e}")

# **Pandas DataFrame に変換**
df = pd.DataFrame(highschool_data)

# **データフレームを表示**
print("\n=== 全高校のデータ ===")
print(df)

# **CSVファイルとして保存（オプション）**
df.to_csv("highschool_data.csv", index=False, encoding="utf-8-sig")
print("\n✅ データを 'highschool_data.csv' に保存しました！")




🏫 https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/sapporominamikoukou/ のデータ取得中...


🏫 https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/sapporokitakoukou/ のデータ取得中...


🏫 https://jyuke-labo.com/koukoujyukentaisaku/hokkaidou/hokuseigakuenzyosikoukou/ のデータ取得中...


=== 全高校のデータ ===
        高校名                                                URL  \
0     札幌南高校  https://jyuke-labo.com/koukoujyukentaisaku/hok...   
1     札幌北高校  https://jyuke-labo.com/koukoujyukentaisaku/hok...   
2  北星学園女子高校  https://jyuke-labo.com/koukoujyukentaisaku/hok...   

                                学科名  学科説明  \
0                               普通科  説明なし   
1                               普通科  説明なし   
2  普通科Ｈｉｇｈコース, 英語科, 普通科Ｃｏｒｅコース, 音楽科  説明なし   

                                          偏差値  \
0                                     ・普通科：73   
1                                     ・普通科：73   
2  ・普通科Ｈｉｇｈコース：61・英語科：59・普通科Ｃｏｒｅコース：52・音楽科：52   

                                                進学実績  
0  北海道大学, 東北大学,